# 🤖 X-MultiVLA: 비트코인 자동 트레이딩 봇
> **VLA(Vision-Language-Action)** 기반 강화학습 트레이딩 에이전트
>
> PatchTST(차트) + FinBERT(뉴스) + Cross-Attention + PPO

---
## 실행 순서
1. 🔧 환경 설치
2. 📁 Google Drive 마운트
3. 📥 데이터 수집
4. 🔄 전처리
5. 🧠 Phase 1: 지도학습 사전 학습
6. 🏋️ Phase 2: 강화학습 (PPO)
7. 📊 백테스트 & 평가
8. 💬 모델 설명 (XAI)

In [ ]:
# ─── CELL 1: 환경 설치 ─────────────────────────────────────
!pip install -q ccxt yfinance fredapi
!pip install -q gymnasium stable-baselines3[extra]
!pip install -q transformers datasets accelerate bitsandbytes
!pip install -q neuralforecast scikit-learn joblib
print('✅ 설치 완료')

In [ ]:
# ─── CELL 2: Google Drive 마운트 ──────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import sys, os
# 프로젝트 루트를 Python 경로에 추가
PROJECT_ROOT = '/content/drive/MyDrive/X-MultiVLA'
os.makedirs(PROJECT_ROOT, exist_ok=True)
sys.path.insert(0, PROJECT_ROOT)

# GPU 확인
import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'💻 Device: {device}')
if device == 'cuda':
    print(f'   GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
# ─── CELL 3: 데이터 수집 ─────────────────────────────────
from data.collector import MarketDataCollector

collector = MarketDataCollector()

# 일봉 2년치 수집 (빠른 테스트)
df_raw = collector.collect_all(timeframe='1d', days=730)
collector.save(df_raw)

print(f'\n수집된 데이터: {df_raw.shape}')
print(df_raw.tail(3))

In [ ]:
# ─── CELL 4: 뉴스 데이터 수집 + 감성 분석 ──────────────
from data.news_fetcher import NewsFetcher

fetcher  = NewsFetcher()
raw_news = fetcher.fetch_raw(pages=10)          # 약 200개 뉴스
news_df  = fetcher.analyze_sentiment(raw_news)

# 차트 데이터와 시간 정렬
df_with_news = fetcher.align_with_chart(df_raw, news_df)
print(f'뉴스 포함 데이터: {df_with_news.shape}')
df_with_news[['news_sentiment_score']].tail()

In [ ]:
# ─── CELL 5: 전처리 ─────────────────────────────────────
import numpy as np
from data.preprocessor import Preprocessor

pp = Preprocessor()
train_ds, val_ds, test_ds = pp.fit_transform(df_with_news)

# Scaler 저장
SCALER_PATH = f'{PROJECT_ROOT}/checkpoints/scaler.pkl'
pp.save_scaler(SCALER_PATH)

print(f'피처 수: {len(pp.feature_cols)}')
print(f'피처 목록: {pp.feature_cols[:8]} ...')

# DataLoader 생성
from torch.utils.data import DataLoader
train_loader = DataLoader(train_ds, batch_size=64, shuffle=True,  num_workers=2)
val_loader   = DataLoader(val_ds,   batch_size=64, shuffle=False, num_workers=2)

In [ ]:
# ─── CELL 6: Phase 1 - 지도학습 사전 학습 ───────────────
import torch
import torch.nn as nn
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from models.vla_agent import VLAPretrainer
from config import train_cfg, data_cfg

NUM_FEATURES = len(pp.feature_cols)
WINDOW       = data_cfg.window_size

pretrain_model = VLAPretrainer(num_features=NUM_FEATURES, window=WINDOW).to(device)
optimizer      = AdamW(pretrain_model.parameters(), lr=train_cfg.pretrain_lr, weight_decay=1e-4)
scheduler      = CosineAnnealingLR(optimizer, T_max=train_cfg.pretrain_epochs)
criterion      = nn.MSELoss()

PRETRAIN_CKPT = f'{PROJECT_ROOT}/checkpoints/pretrain_best.pt'
os.makedirs(os.path.dirname(PRETRAIN_CKPT), exist_ok=True)

best_val_loss = float('inf')
train_losses, val_losses = [], []

for epoch in range(train_cfg.pretrain_epochs):
    # ── Train ──
    pretrain_model.train()
    t_loss = 0
    for x_seq, y_true in train_loader:
        x_seq  = x_seq.to(device)          # (B, window, F)
        y_true = y_true.to(device)         # (B,)

        # 뉴스 벡터: 피처 중 news 컬럼 추출 (없으면 zeros)
        news_idx = [pp.feature_cols.index(c) for c in ['news_pos','news_neg','news_neu','news_sentiment_score'] if c in pp.feature_cols]
        if len(news_idx) == 4:
            news_vec = x_seq[:, -1, news_idx]   # 마지막 타임스텝의 뉴스 벡터
        else:
            news_vec = torch.zeros(x_seq.size(0), 4, device=device)

        pred   = pretrain_model(x_seq, news_vec).squeeze()
        loss   = criterion(pred, y_true)
        optimizer.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(pretrain_model.parameters(), 1.0)
        optimizer.step()
        t_loss += loss.item()

    # ── Validation ──
    pretrain_model.eval()
    v_loss = 0
    with torch.no_grad():
        for x_seq, y_true in val_loader:
            x_seq  = x_seq.to(device)
            y_true = y_true.to(device)
            if len(news_idx) == 4:
                news_vec = x_seq[:, -1, news_idx]
            else:
                news_vec = torch.zeros(x_seq.size(0), 4, device=device)
            pred  = pretrain_model(x_seq, news_vec).squeeze()
            v_loss += criterion(pred, y_true).item()

    t_loss /= len(train_loader)
    v_loss /= len(val_loader)
    train_losses.append(t_loss)
    val_losses.append(v_loss)
    scheduler.step()

    if v_loss < best_val_loss:
        best_val_loss = v_loss
        torch.save(pretrain_model.state_dict(), PRETRAIN_CKPT)

    if (epoch + 1) % 5 == 0:
        print(f'Epoch {epoch+1:3d}/{train_cfg.pretrain_epochs} | Train: {t_loss:.6f} | Val: {v_loss:.6f}')

print(f'\n✅ 사전 학습 완료. Best Val Loss: {best_val_loss:.6f}')

# 학습 곡선 시각화
import matplotlib.pyplot as plt
plt.figure(figsize=(10,4))
plt.plot(train_losses, label='Train Loss')
plt.plot(val_losses,   label='Val Loss')
plt.xlabel('Epoch'); plt.ylabel('MSE Loss')
plt.title('Phase 1: 사전 학습 손실 곡선')
plt.legend(); plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# ─── CELL 7: Phase 2 - PPO 강화학습 ─────────────────────
from stable_baselines3.common.callbacks import EvalCallback, CheckpointCallback
from stable_baselines3.common.vec_env import DummyVecEnv
from utils.gym_env import CryptoTradingEnv
from models.vla_agent import VLAAgent

# 사전 학습 모델 로드
pretrain_model.load_state_dict(torch.load(PRETRAIN_CKPT, map_location=device))

# 학습 / 검증 데이터 배열 (numpy)
df_eng      = pp.engineer_features(df_with_news)
X_all       = pp.scaler.transform(df_eng[pp.feature_cols].values)
prices_all  = df_eng['BTC_close'].values

n_train     = int(len(X_all) * data_cfg.train_ratio)
X_train_np  = X_all[:n_train]
p_train_np  = prices_all[:n_train]
X_val_np    = X_all[n_train:]
p_val_np    = prices_all[n_train:]

# Gym 환경 생성
def make_train_env():
    return CryptoTradingEnv(X_train_np, p_train_np, window=WINDOW)

def make_eval_env():
    return CryptoTradingEnv(X_val_np, p_val_np, window=WINDOW)

train_env   = DummyVecEnv([make_train_env])
eval_env    = DummyVecEnv([make_eval_env])

RL_CKPT     = f'{PROJECT_ROOT}/checkpoints/ppo_vla'
callbacks   = [
    EvalCallback(eval_env, best_model_save_path=f'{PROJECT_ROOT}/checkpoints',
                 log_path=f'{PROJECT_ROOT}/logs', eval_freq=5000, verbose=0),
    CheckpointCallback(save_freq=10000, save_path=f'{PROJECT_ROOT}/checkpoints', verbose=0),
]

# VLA 에이전트 생성 & 학습
agent = VLAAgent(
    env=train_env,
    num_features=NUM_FEATURES,
    window=WINDOW,
    pretrained_model=pretrain_model,
    device=device,
)

agent.train(total_timesteps=200_000, callback=callbacks)   # Colab 테스트용 20만 스텝
agent.save(RL_CKPT)
print(f'✅ PPO 학습 완료 → {RL_CKPT}')

In [ ]:
# ─── CELL 8: 백테스트 & 성과 분석 ───────────────────────
from utils.evaluator import Backtester

# 테스트 데이터
n_val   = int(len(X_all) * data_cfg.val_ratio)
X_test  = X_all[n_train + n_val:]
p_test  = prices_all[n_train + n_val:]

test_env = CryptoTradingEnv(X_test, p_test, window=WINDOW)

# 에이전트 로드 (또는 위 셀의 agent 직접 사용)
from models.vla_agent import VLAAgent
agent = VLAAgent.load(RL_CKPT + '.zip', env=DummyVecEnv([lambda: CryptoTradingEnv(X_test, p_test, window=WINDOW)]))

bt = Backtester(agent, X_test, p_test, window=WINDOW)
result = bt.run()

bt.print_metrics(result)
bt.plot(result, save_path=f'{PROJECT_ROOT}/outputs/backtest_result.png')

In [ ]:
# ─── CELL 9: XAI - 모델 설명 생성 ───────────────────────
from models.xai_explainer import VLAExplainer

explainer = VLAExplainer(pretrain_model, pp.feature_cols)

# 테스트 셋의 첫 번째 샘플로 설명 생성
sample_chart = torch.tensor(X_test[WINDOW:WINDOW+1], dtype=torch.float32).unsqueeze(0).to(device)
# (1, window, F) 형태로
sample_chart = torch.tensor(X_test[:WINDOW], dtype=torch.float32).unsqueeze(0).to(device)

obs, _ = test_env.reset()
action, _ = agent.predict(obs)

news_score = float(X_test[WINDOW - 1, pp.feature_cols.index('news_sentiment_score')]) if 'news_sentiment_score' in pp.feature_cols else 0.0

explanation = explainer.explain(
    chart_seq  = sample_chart,
    news_score = news_score,
    action     = int(action),
)
print(explanation)

In [ ]:
# ─── CELL 10: 피처 중요도 분석 ──────────────────────────
# 작은 배치로 permutation importance 계산
N = min(100, len(X_test) - WINDOW)
chart_batch = torch.tensor(
    np.array([X_test[i:i+WINDOW] for i in range(N)]), dtype=torch.float32
).to(device)
news_batch = torch.zeros(N, 4, device=device)

df_imp = explainer.feature_importance(chart_batch, news_batch, n_repeat=3, top_k=12)
print(df_imp.head(12).to_string(index=False))